# Modul 14: Vorwärtsrechnung, Backpropagation, MLPs und Faltungen

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Vorwärts und Rückwärts, MLP und Faltungen  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene NumPy-Anwendung mit mathematischem Denken  
    **Orientierungszeit:** etwa 150 bis 210 Minuten

    ## Überblick

    Sie implementieren zentrale Bausteine neuronaler Netze mit NumPy. Von einem künstlichen Neuron über stabile Softmax- und Verlustberechnungen führt der Weg zu Backpropagation, Mini-Batch-Training und einfachen CNN-Operationen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_14A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_14B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Neuronen, Dense-Schichten und Aktivierungsfunktionen mit NumPy berechnen.
- Eine numerisch stabile Vorwärtsrechnung mit Softmax und Kreuzentropie umsetzen.
- Gradienten mit der Kettenregel herleiten und numerisch kontrollieren.
- Ein kleines MLP mit Mini-Batches, Momentum und L2-Regularisierung trainieren.
- Overfitting mit Validierungsdaten und Early Stopping erkennen.
- Faltung, Padding, Pooling und die resultierenden Tensorformen nachvollziehen.

    ## Bewertete Fähigkeiten

    - Matrixmultiplikation und Aktivierungsfunktionen
- stabile Softmax- und Kreuzentropieberechnung
- Backpropagation und numerische Gradientenprüfung
- Mini-Batch-SGD, Momentum, L2 und Early Stopping
- zweidimensionale Faltung, Padding, Max-Pooling und Formplanung

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Ein kleiner nichtlinearer Datensatz reicht aus, um den Nutzen einer
# verborgenen Schicht zu untersuchen. Alle Daten entstehen lokal.
X_14, y_14 = make_moons(n_samples=480, noise=0.22, random_state=RANDOM_SEED)
X_train_14, X_temp_14, y_train_14, y_temp_14 = train_test_split(
    X_14,
    y_14,
    test_size=0.40,
    stratify=y_14,
    random_state=RANDOM_SEED,
)
X_valid_14, X_test_14, y_valid_14, y_test_14 = train_test_split(
    X_temp_14,
    y_temp_14,
    test_size=0.50,
    stratify=y_temp_14,
    random_state=RANDOM_SEED,
)

# Die Skalierung wird ausschließlich mit den Trainingsdaten gelernt.
scaler_14 = StandardScaler()
X_train_14 = scaler_14.fit_transform(X_train_14)
X_valid_14 = scaler_14.transform(X_valid_14)
X_test_14 = scaler_14.transform(X_test_14)

# Ein kleines Graustufenbild und ein Kantenfilter dienen den CNN-Aufgaben.
image_14 = np.array(
    [
        [0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 0],
        [0, 1, 0, 1, 0, 0],
        [0, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
    ],
    dtype=np.float64,
)
vertical_edge_kernel_14 = np.array(
    [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    dtype=np.float64,
)

print("Train/Valid/Test:", X_train_14.shape, X_valid_14.shape, X_test_14.shape)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Neuron, Dense-Schicht und Aktivierungen

    Implementieren Sie die Bausteine einer kleinen Dense-Schicht.

1. Schreiben Sie Funktionen für `sigmoid`, `tanh` und `relu`.
2. Berechnen Sie für einen einzelnen Eingabevektor zunächst den linearen Wert eines Neurons und anschließend seine Sigmoid-Ausgabe.
3. Berechnen Sie für einen Stapel aus drei Beispielen die Ausgabe einer Dense-Schicht mit drei Neuronen.
4. Wenden Sie alle drei Aktivierungsfunktionen auf dieselben Voraktivierungen an.
5. Prüfen Sie die Formen mit sinnvollen `assert`-Anweisungen und erklären Sie, warum eine reine Verkettung linearer Schichten ohne nichtlineare Aktivierung keine zusätzliche Ausdruckskraft erzeugt.

> **Hinweis:** Schreiben Sie zuerst die erwarteten Formen neben jede Matrixmultiplikation.

In [ ]:
single_input = np.array([0.8, -1.2, 0.5])
neuron_weights = np.array([0.4, -0.6, 0.2])
neuron_bias = -0.1

X_batch = np.array(
    [
        [0.8, -1.2, 0.5],
        [1.0, 0.2, -0.4],
        [-0.5, 1.5, 0.7],
    ]
)
dense_weights = np.array(
    [
        [0.3, -0.2, 0.5],
        [-0.7, 0.4, 0.1],
        [0.2, 0.6, -0.3],
    ]
)
dense_bias = np.array([0.1, -0.2, 0.05])

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Neuron, Dense-Schicht und Aktivierungen
#
# Ziel dieser Codezelle:
# Implementieren Sie die Bausteine einer kleinen Dense-Schicht. 1. Schreiben Sie
# Funktionen für sigmoid, tanh und relu. 2. Berechnen Sie für einen einzelnen
# Eingabevektor zunächst den linearen Wert eines Neurons und ans...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

single_input = np.array([0.8, -1.2, 0.5])
neuron_weights = np.array([0.4, -0.6, 0.2])
neuron_bias = -0.1

X_batch = np.array(
    [
        [0.8, -1.2, 0.5],
        [1.0, 0.2, -0.4],
        [-0.5, 1.5, 0.7],
    ]
)
dense_weights = np.array(
    [
        [0.3, -0.2, 0.5],
        [-0.7, 0.4, 0.1],
        [0.2, 0.6, -0.3],
    ]
)
dense_bias = np.array([0.1, -0.2, 0.05])

# Sigmoid bildet beliebige reelle Werte auf das offene Intervall
# zwischen null und eins ab. np.asarray erlaubt Skalare und Arrays.
def sigmoid(values):
    values = np.asarray(values, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-values))

# NumPy stellt tanh bereits stabil und vektorisiert bereit.
def tanh_activation(values):
    return np.tanh(np.asarray(values, dtype=np.float64))

# ReLU lässt positive Werte unverändert und setzt negative auf null.
def relu(values):
    return np.maximum(0.0, np.asarray(values, dtype=np.float64))

# Ein Neuron berechnet zuerst eine gewichtete Summe plus Bias.
neuron_linear = single_input @ neuron_weights + neuron_bias
neuron_output = sigmoid(neuron_linear)

# Bei einer Dense-Schicht sind die Beispiele die Zeilen von X_batch.
# Jede Spalte der Gewichtsmatrix gehört zu einem Ausgabeneuron.
dense_linear = X_batch @ dense_weights + dense_bias
dense_sigmoid = sigmoid(dense_linear)
dense_tanh = tanh_activation(dense_linear)
dense_relu = relu(dense_linear)

# Frühe Formprüfungen machen vertauschte Achsen sichtbar.
assert single_input.shape == neuron_weights.shape
assert X_batch.shape[1] == dense_weights.shape[0]
assert dense_linear.shape == (3, 3)
assert dense_relu.shape == dense_linear.shape

print("Lineare Neuronenausgabe:", round(float(neuron_linear), 4))
print("Sigmoid-Neuronenausgabe:", round(float(neuron_output), 4))
print("Dense-Voraktivierungen:\n", np.round(dense_linear, 3))
print("Sigmoid:\n", np.round(dense_sigmoid, 3))
print("Tanh:\n", np.round(dense_tanh, 3))
print("ReLU:\n", np.round(dense_relu, 3))

### Reflexion zu Aufgabe 1

Jede lineare Schicht führt eine affine Transformation aus. Werden mehrere solche Transformationen ohne nichtlineare Aktivierung verkettet, lassen sie sich algebraisch zu genau einer affinen Transformation zusammenfassen. Nichtlinearitäten wie ReLU, Sigmoid oder Tanh ermöglichen dagegen gekrümmte Entscheidungsgrenzen und komplexere Repräsentationen. Die Aktivierung muss zur Aufgabe und zur Position im Netz passen: ReLU ist häufig in verborgenen Schichten nützlich, während Sigmoid oft eine binäre Ausgabewahrscheinlichkeit darstellt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Stabile Vorwärtsrechnung und Kreuzentropie

    Bauen Sie eine vollständige Vorwärtsrechnung für ein zweischichtiges Mehrklassen-Netz.

1. Implementieren Sie eine zeilenweise, numerisch stabile Softmax-Funktion.
2. Berechnen Sie `X @ W1 + b1`, wenden Sie ReLU an und erzeugen Sie anschließend die Logits der zweiten Schicht.
3. Wandeln Sie die Logits in Klassenwahrscheinlichkeiten um.
4. Berechnen Sie die mittlere Kreuzentropie für ganzzahlige Klassenlabels.
5. Vergleichen Sie die stabile Softmax mit einer naiven Variante bei sehr großen Logits. Prüfen Sie, dass jede Wahrscheinlichkeitszeile ungefähr eins ergibt.

> **Hinweis:** Verwenden Sie bei allen zeilenweisen Operationen `axis=1` und `keepdims=True`.

In [ ]:
X_forward = np.array(
    [[1.0, -0.5], [0.2, 1.4], [-1.2, 0.7], [0.5, 0.3]],
    dtype=np.float64,
)
y_forward = np.array([0, 2, 1, 0])

W1 = np.array([[0.6, -0.4, 0.2], [-0.3, 0.8, 0.5]])
b1 = np.array([0.1, -0.1, 0.0])
W2 = np.array([[0.5, -0.2, 0.1], [-0.4, 0.7, 0.2], [0.3, -0.1, 0.6]])
b2 = np.array([0.0, 0.1, -0.1])

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Stabile Vorwärtsrechnung und Kreuzentropie
#
# Ziel dieser Codezelle:
# Bauen Sie eine vollständige Vorwärtsrechnung für ein zweischichtiges Mehrklassen-
# Netz. 1. Implementieren Sie eine zeilenweise, numerisch stabile Softmax-Funktion.
# 2. Berechnen Sie X @ W1 + b1, wenden Sie ReLU an und e...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_forward = np.array(
    [[1.0, -0.5], [0.2, 1.4], [-1.2, 0.7], [0.5, 0.3]],
    dtype=np.float64,
)
y_forward = np.array([0, 2, 1, 0])

W1 = np.array([[0.6, -0.4, 0.2], [-0.3, 0.8, 0.5]])
b1 = np.array([0.1, -0.1, 0.0])
W2 = np.array([[0.5, -0.2, 0.1], [-0.4, 0.7, 0.2], [0.3, -0.1, 0.6]])
b2 = np.array([0.0, 0.1, -0.1])

def stable_softmax(logits):
    # Das Zeilenmaximum wird abgezogen. Diese Verschiebung verändert
    # die resultierenden Wahrscheinlichkeiten nicht, verhindert aber
    # sehr große Exponentialwerte und damit numerischen Überlauf.
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)

def sparse_cross_entropy(probabilities, class_labels):
    # Für jedes Beispiel wird nur die Wahrscheinlichkeit der wahren
    # Klasse ausgewählt. Clipping schützt log(0), ohne die üblichen
    # Werte merklich zu verändern.
    row_indices = np.arange(len(class_labels))
    correct_probabilities = probabilities[row_indices, class_labels]
    safe_probabilities = np.clip(correct_probabilities, 1e-12, 1.0)
    return -np.mean(np.log(safe_probabilities))

# Erste affine Transformation und nichtlineare Aktivierung.
hidden_linear = X_forward @ W1 + b1
hidden_activation = np.maximum(0.0, hidden_linear)

# Die zweite Schicht erzeugt pro Beispiel einen Score je Klasse.
logits = hidden_activation @ W2 + b2
probabilities = stable_softmax(logits)
loss = sparse_cross_entropy(probabilities, y_forward)
predictions = np.argmax(probabilities, axis=1)

# Form- und Wahrscheinlichkeitsprüfungen erkennen typische Fehler.
assert hidden_activation.shape == (4, 3)
assert logits.shape == (4, 3)
assert np.allclose(probabilities.sum(axis=1), 1.0)

# Die naive Softmax läuft bei großen Logits über. errstate hält die
# Demonstration kontrolliert, statt das Notebook abzubrechen.
large_logits = np.array([[1000.0, 1001.0, 999.0]])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(large_logits) / np.exp(large_logits).sum(axis=1, keepdims=True)
stable = stable_softmax(large_logits)

print("Logits:\n", np.round(logits, 3))
print("Wahrscheinlichkeiten:\n", np.round(probabilities, 3))
print("Vorhersagen:", predictions.tolist())
print("Mittlere Kreuzentropie:", round(float(loss), 4))
print("Naive Softmax bei großen Logits:", naive)
print("Stabile Softmax:", np.round(stable, 6))

### Reflexion zu Aufgabe 2

Softmax darf für jedes Beispiel nur entlang der Klassenachse normalisieren. Das Abziehen des Zeilenmaximums ist mathematisch zulässig, weil sich ein gemeinsamer Faktor im Zähler und Nenner kürzt. Kreuzentropie bestraft besonders stark, wenn das Modell der richtigen Klasse eine sehr kleine Wahrscheinlichkeit zuweist. Die Logits selbst sind noch keine Wahrscheinlichkeiten und sollten nicht direkt als solche interpretiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Backpropagation prüfen und einen Lernschritt ausführen

    Für eine Dense-Schicht mit mittlerem quadratischem Fehler sollen Sie Vorwärts- und Rückwärtsrechnung kontrollieren.

1. Berechnen Sie Vorhersagen und MSE.
2. Leiten Sie die Gradienten nach Gewichten, Bias und Eingaben mit der Kettenregel ab.
3. Schreiben Sie eine Funktion, die den Verlust für beliebige Gewichte berechnet.
4. Prüfen Sie **alle** Gewichtselemente mit einem zentralen Differenzenquotienten und berichten Sie die größte absolute Abweichung.
5. Führen Sie einen Gradientenabstiegsschritt aus und bestätigen Sie, dass der Verlust sinkt.

> **Hinweis:** Achten Sie darauf, ob der MSE über Beispiele oder über alle Ausgabeelemente gemittelt wird.

In [ ]:
X_gradient = np.array(
    [[0.2, -0.4], [0.7, 0.1], [-0.3, 0.5], [1.0, -0.2]],
    dtype=np.float64,
)
y_gradient = np.array(
    [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]],
    dtype=np.float64,
)
W_gradient = np.array([[0.3, -0.1], [-0.2, 0.4]], dtype=np.float64)
b_gradient = np.array([[0.05, -0.05]], dtype=np.float64)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Backpropagation prüfen und einen Lernschritt ausführen
#
# Ziel dieser Codezelle:
# Für eine Dense-Schicht mit mittlerem quadratischem Fehler sollen Sie Vorwärts- und
# Rückwärtsrechnung kontrollieren. 1. Berechnen Sie Vorhersagen und MSE. 2. Leiten
# Sie die Gradienten nach Gewichten, Bias und Eingaben...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_gradient = np.array(
    [[0.2, -0.4], [0.7, 0.1], [-0.3, 0.5], [1.0, -0.2]],
    dtype=np.float64,
)
y_gradient = np.array(
    [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]],
    dtype=np.float64,
)
W_gradient = np.array([[0.3, -0.1], [-0.2, 0.4]], dtype=np.float64)
b_gradient = np.array([[0.05, -0.05]], dtype=np.float64)

def dense_mse_loss(current_weights, current_bias):
    # Eine reine Verlustfunktion ist praktisch für numerische Tests.
    current_predictions = X_gradient @ current_weights + current_bias
    return np.mean((current_predictions - y_gradient) ** 2)

# Vorwärtslauf der Dense-Schicht.
predictions = X_gradient @ W_gradient + b_gradient
errors = predictions - y_gradient
loss_before = np.mean(errors ** 2)

# Da MSE über alle Ausgabeelemente mittelt, wird durch die gesamte
# Elementzahl geteilt. Dieser Faktor ist eine häufige Fehlerquelle.
grad_predictions = 2.0 * errors / errors.size

# Die Kettenregel verteilt den eingehenden Gradienten auf Parameter
# und Eingaben. Die Matrixformen folgen dem Vorwärtslauf rückwärts.
grad_weights = X_gradient.T @ grad_predictions
grad_bias = np.sum(grad_predictions, axis=0, keepdims=True)
grad_inputs = grad_predictions @ W_gradient.T

# Zentraler Differenzenquotient für jedes Gewicht. Er benötigt zwei
# Vorwärtsläufe pro Parameter und dient nur als Diagnosewerkzeug.
epsilon = 1e-5
numeric_grad_weights = np.zeros_like(W_gradient)
for row in range(W_gradient.shape[0]):
    for column in range(W_gradient.shape[1]):
        weights_plus = W_gradient.copy()
        weights_minus = W_gradient.copy()
        weights_plus[row, column] += epsilon
        weights_minus[row, column] -= epsilon
        loss_plus = dense_mse_loss(weights_plus, b_gradient)
        loss_minus = dense_mse_loss(weights_minus, b_gradient)
        numeric_grad_weights[row, column] = (loss_plus - loss_minus) / (2.0 * epsilon)

max_difference = np.max(np.abs(grad_weights - numeric_grad_weights))

# Ein kontrollierter Gradientenschritt sollte lokal den Verlust senken.
learning_rate = 0.2
updated_weights = W_gradient - learning_rate * grad_weights
updated_bias = b_gradient - learning_rate * grad_bias
loss_after = dense_mse_loss(updated_weights, updated_bias)

assert grad_weights.shape == W_gradient.shape
assert grad_bias.shape == b_gradient.shape
assert grad_inputs.shape == X_gradient.shape
assert max_difference < 1e-7
assert loss_after < loss_before

print("Analytische Gradienten:\n", np.round(grad_weights, 7))
print("Numerische Gradienten:\n", np.round(numeric_grad_weights, 7))
print("Größte absolute Abweichung:", f"{max_difference:.2e}")
print("Verlust vorher:", round(float(loss_before), 6))
print("Verlust nach einem Schritt:", round(float(loss_after), 6))

### Reflexion zu Aufgabe 3

Eine numerische Gradientenprüfung ist langsam, aber sehr hilfreich, um Vorzeichen-, Skalierungs- und Formfehler in einer selbst geschriebenen Backpropagation zu finden. Eine kleine Abweichung ist wegen Gleitkommaarithmetik normal. Ein bestandener Gradiententest garantiert jedoch nicht automatisch ein gutes Modell. Auch Datenaufteilung, Lernrate, Architektur und Trainingslogik müssen stimmen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Ein MLP mit Mini-Batches, Momentum und Early Stopping trainieren

    Trainieren Sie ein zweischichtiges binäres MLP auf den vorbereiteten Moon-Daten.

1. Initialisieren Sie kleine Gewichte für eine Architektur `2 -> 12 -> 1`.
2. Implementieren Sie Vorwärtsrechnung, binäre Kreuzentropie und Backpropagation.
3. Trainieren Sie mit gemischten Mini-Batches, SGD mit Momentum und L2-Regularisierung.
4. Speichern Sie pro Epoche Trainings- und Validierungsverlust.
5. Implementieren Sie Early Stopping mit Wiederherstellung der besten Parameter.
6. Berichten Sie Train-, Validierungs- und Testgenauigkeit und visualisieren Sie Verlustkurven sowie die Entscheidungsgrenze.

Verwenden Sie das Testset erst nach Abschluss aller Modellentscheidungen.

> **Hinweis:** Testen Sie die Vorwärtsrechnung zuerst auf zwei Beispielen, bevor Sie die Trainingsschleife starten.

In [ ]:
# Empfohlene Startwerte. Sie dürfen die Lernrate vorsichtig anpassen.
hidden_units = 12
max_epochs = 500
batch_size = 32
learning_rate = 0.04
momentum = 0.9
l2_strength = 0.001
patience = 45

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Ein MLP mit Mini-Batches, Momentum und Early Stopping trainieren
#
# Ziel dieser Codezelle:
# Trainieren Sie ein zweischichtiges binäres MLP auf den vorbereiteten Moon-Daten.
# 1. Initialisieren Sie kleine Gewichte für eine Architektur 2 - 12 - 1. 2.
# Implementieren Sie Vorwärtsrechnung, binäre Kreuzentropie und...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

hidden_units = 12
max_epochs = 500
batch_size = 32
learning_rate = 0.04
momentum = 0.9
l2_strength = 0.001
patience = 45

# Ein eigener Zufallszahlengenerator hält Initialisierung und
# Mini-Batch-Reihenfolge reproduzierbar.
training_rng = np.random.default_rng(RANDOM_SEED)
input_units = X_train_14.shape[1]

# Xavier-artige Skalen verhindern sehr große Startaktivierungen.
W1 = training_rng.normal(0.0, np.sqrt(1.0 / input_units), size=(input_units, hidden_units))
b1 = np.zeros((1, hidden_units))
W2 = training_rng.normal(0.0, np.sqrt(1.0 / hidden_units), size=(hidden_units, 1))
b2 = np.zeros((1, 1))

# Momentum benötigt für jeden Parameter einen Geschwindigkeitswert.
velocity_W1 = np.zeros_like(W1)
velocity_b1 = np.zeros_like(b1)
velocity_W2 = np.zeros_like(W2)
velocity_b2 = np.zeros_like(b2)

def sigmoid_stable(values):
    # Clipping schützt exp vor sehr großen Beträgen.
    clipped = np.clip(values, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-clipped))

def mlp_forward(features, parameters):
    current_W1, current_b1, current_W2, current_b2 = parameters
    hidden_linear = features @ current_W1 + current_b1
    hidden = np.maximum(0.0, hidden_linear)
    output_linear = hidden @ current_W2 + current_b2
    probabilities = sigmoid_stable(output_linear)
    cache = (features, hidden_linear, hidden, probabilities)
    return probabilities, cache

def binary_loss(features, labels, parameters):
    probabilities, _ = mlp_forward(features, parameters)
    labels_column = labels.reshape(-1, 1)
    probabilities = np.clip(probabilities, 1e-10, 1.0 - 1e-10)
    data_loss = -np.mean(
        labels_column * np.log(probabilities)
        + (1.0 - labels_column) * np.log(1.0 - probabilities)
    )
    # Nur Gewichtsmatrizen werden mit L2 bestraft, nicht die Biases.
    current_W1, _, current_W2, _ = parameters
    penalty = 0.5 * l2_strength * (
        np.sum(current_W1 ** 2) + np.sum(current_W2 ** 2)
    )
    return float(data_loss + penalty)

train_losses = []
valid_losses = []
best_valid_loss = np.inf
best_parameters = None
epochs_without_improvement = 0

for epoch in range(max_epochs):
    # Shuffling erzeugt in jeder Epoche neue, aber reproduzierbare
    # Mini-Batch-Zusammenstellungen.
    shuffled_indices = training_rng.permutation(len(X_train_14))

    for start in range(0, len(shuffled_indices), batch_size):
        batch_indices = shuffled_indices[start:start + batch_size]
        X_batch = X_train_14[batch_indices]
        y_batch = y_train_14[batch_indices].reshape(-1, 1)

        parameters = (W1, b1, W2, b2)
        probabilities, cache = mlp_forward(X_batch, parameters)
        batch_features, hidden_linear, hidden, _ = cache
        current_batch_size = len(X_batch)

        # Bei Sigmoid plus binärer Kreuzentropie vereinfacht sich der
        # Gradient nach dem Ausgangslogit zu p minus y.
        grad_output_linear = (probabilities - y_batch) / current_batch_size
        grad_W2 = hidden.T @ grad_output_linear + l2_strength * W2
        grad_b2 = np.sum(grad_output_linear, axis=0, keepdims=True)

        grad_hidden = grad_output_linear @ W2.T
        grad_hidden_linear = grad_hidden * (hidden_linear > 0.0)
        grad_W1 = batch_features.T @ grad_hidden_linear + l2_strength * W1
        grad_b1 = np.sum(grad_hidden_linear, axis=0, keepdims=True)

        # Momentum glättet aufeinanderfolgende Update-Richtungen.
        velocity_W1 = momentum * velocity_W1 - learning_rate * grad_W1
        velocity_b1 = momentum * velocity_b1 - learning_rate * grad_b1
        velocity_W2 = momentum * velocity_W2 - learning_rate * grad_W2
        velocity_b2 = momentum * velocity_b2 - learning_rate * grad_b2

        W1 += velocity_W1
        b1 += velocity_b1
        W2 += velocity_W2
        b2 += velocity_b2

    parameters = (W1, b1, W2, b2)
    train_loss = binary_loss(X_train_14, y_train_14, parameters)
    valid_loss = binary_loss(X_valid_14, y_valid_14, parameters)
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    # Die Validierungsdaten steuern Early Stopping. Das Testset wird
    # während dieses Prozesses bewusst nicht verwendet.
    if valid_loss < best_valid_loss - 1e-5:
        best_valid_loss = valid_loss
        best_parameters = tuple(parameter.copy() for parameter in parameters)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        break

# Die Parameter mit dem besten Validierungsverlust werden restauriert.
W1, b1, W2, b2 = best_parameters
best_parameters = (W1, b1, W2, b2)

def predict_mlp(features):
    probabilities, _ = mlp_forward(features, best_parameters)
    return (probabilities.ravel() >= 0.5).astype(int)

train_accuracy = accuracy_score(y_train_14, predict_mlp(X_train_14))
valid_accuracy = accuracy_score(y_valid_14, predict_mlp(X_valid_14))
test_accuracy = accuracy_score(y_test_14, predict_mlp(X_test_14))

print("Trainierte Epochen:", len(train_losses))
print("Beste Validierungs-BCE inklusive L2:", round(best_valid_loss, 4))
print("Train-Genauigkeit:", round(train_accuracy, 3))
print("Validierungsgenauigkeit:", round(valid_accuracy, 3))
print("Testgenauigkeit:", round(test_accuracy, 3))

# Lernkurven zeigen Optimierung und mögliche Generalisierungslücken.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Training")
ax.plot(valid_losses, label="Validierung")
ax.set_title("MLP-Verlustkurven")
ax.set_xlabel("Epoche")
ax.set_ylabel("Binäre Kreuzentropie plus L2")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Ein Gitter macht die nichtlineare Entscheidungsgrenze sichtbar.
x_min, x_max = X_train_14[:, 0].min() - 0.8, X_train_14[:, 0].max() + 0.8
y_min, y_max = X_train_14[:, 1].min() - 0.8, X_train_14[:, 1].max() + 0.8
grid_x, grid_y = np.meshgrid(
    np.linspace(x_min, x_max, 180),
    np.linspace(y_min, y_max, 180),
)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
grid_probabilities, _ = mlp_forward(grid_points, best_parameters)

fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(
    grid_x,
    grid_y,
    grid_probabilities.reshape(grid_x.shape),
    levels=np.linspace(0.0, 1.0, 11),
    alpha=0.35,
)
ax.scatter(X_test_14[:, 0], X_test_14[:, 1], c=y_test_14, edgecolor="black")
ax.set_title("Entscheidungsgrenze auf dem zurückgehaltenen Testset")
ax.set_xlabel("skaliertes Merkmal 1")
ax.set_ylabel("skaliertes Merkmal 2")
fig.colorbar(contour, ax=ax, label="P(Klasse 1)")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 4

Das MLP kombiniert eine nichtlineare verborgene Repräsentation mit einer probabilistischen Ausgabe. Mini-Batches reduzieren den Speicherbedarf und liefern häufigere Updates. Momentum nutzt vorherige Update-Richtungen, während L2 große Gewichte begrenzt. Early Stopping ist nur dann sauber, wenn eine getrennte Validierungsmenge die Auswahl steuert und das Testset bis zur endgültigen Bewertung unangetastet bleibt. Unterschiede zwischen Trainings-, Validierungs- und Testwerten sollten gemeinsam betrachtet werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Faltung, Padding, Pooling und Tensorformen

    Implementieren und untersuchen Sie grundlegende CNN-Operationen ohne Deep-Learning-Framework.

1. Schreiben Sie eine Funktion `conv2d_single_channel(image, kernel, padding=0, stride=1)` für Kreuzkorrelation, wie sie in CNNs üblich ist.
2. Berechnen Sie die Ausgabe für das vorbereitete Bild einmal ohne Padding und einmal mit `padding=1`.
3. Schreiben Sie eine Funktion für `2 x 2` Max-Pooling mit Stride 2.
4. Vergleichen Sie berechnete und theoretisch erwartete Höhen und Breiten.
5. Planen Sie die Formen für einen Batch `(16, 28, 28, 1)` nach `Conv2D(8, 3, padding="same")`, `MaxPool2D(2)`, `Conv2D(16, 3, padding="valid")`, `MaxPool2D(2)` und anschließendem Flatten.

> **Hinweis:** Berechnen Sie zuerst nur die Ausgabehöhe. Für die Breite gilt dieselbe Formel.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Faltung, Padding, Pooling und Tensorformen
#
# Ziel dieser Codezelle:
# Implementieren und untersuchen Sie grundlegende CNN-Operationen ohne Deep-
# Learning-Framework. 1. Schreiben Sie eine Funktion conv2dsinglechannel(image,
# kernel, padding=0, stride=1) für Kreuzkorrelation, wie sie in CNN...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def output_size(input_size, kernel_size, padding=0, stride=1):
    # Diese ganzzahlige Formel gilt für eine einzelne räumliche Achse.
    return (input_size + 2 * padding - kernel_size) // stride + 1

def conv2d_single_channel(image, kernel, padding=0, stride=1):
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)

    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError("Bild und Kernel müssen zweidimensional sein.")
    if stride < 1 or padding < 0:
        raise ValueError("Stride muss positiv und Padding nichtnegativ sein.")

    # np.pad ergänzt auf allen Seiten Nullen. Bei padding=0 bleibt
    # das Bild unverändert.
    padded = np.pad(image, pad_width=padding, mode="constant")
    kernel_height, kernel_width = kernel.shape
    output_height = output_size(image.shape[0], kernel_height, padding, stride)
    output_width = output_size(image.shape[1], kernel_width, padding, stride)
    result = np.empty((output_height, output_width), dtype=np.float64)

    # In CNNs wird der Kernel üblicherweise nicht gespiegelt. Diese
    # Operation heißt mathematisch Kreuzkorrelation.
    for out_row in range(output_height):
        for out_col in range(output_width):
            row_start = out_row * stride
            col_start = out_col * stride
            patch = padded[
                row_start:row_start + kernel_height,
                col_start:col_start + kernel_width,
            ]
            result[out_row, out_col] = np.sum(patch * kernel)
    return result

def max_pool_2x2(feature_map):
    feature_map = np.asarray(feature_map, dtype=np.float64)
    pool_size = 2
    stride = 2
    output_height = output_size(feature_map.shape[0], pool_size, 0, stride)
    output_width = output_size(feature_map.shape[1], pool_size, 0, stride)
    pooled = np.empty((output_height, output_width), dtype=np.float64)

    for row in range(output_height):
        for column in range(output_width):
            patch = feature_map[
                row * stride:row * stride + pool_size,
                column * stride:column * stride + pool_size,
            ]
            pooled[row, column] = np.max(patch)
    return pooled

valid_feature_map = conv2d_single_channel(
    image_14,
    vertical_edge_kernel_14,
    padding=0,
    stride=1,
)
same_feature_map = conv2d_single_channel(
    image_14,
    vertical_edge_kernel_14,
    padding=1,
    stride=1,
)
pooled_feature_map = max_pool_2x2(same_feature_map)

# Theoretische und tatsächliche Formen müssen übereinstimmen.
assert valid_feature_map.shape == (
    output_size(6, 3, 0, 1),
    output_size(6, 3, 0, 1),
)
assert same_feature_map.shape == (6, 6)
assert pooled_feature_map.shape == (3, 3)

print("Valid-Feature-Map, Form", valid_feature_map.shape)
print(np.round(valid_feature_map, 2))
print("Same-Feature-Map, Form", same_feature_map.shape)
print(np.round(same_feature_map, 2))
print("Nach 2x2-Max-Pooling, Form", pooled_feature_map.shape)
print(np.round(pooled_feature_map, 2))

# Formplanung im channels-last-Format von Keras:
# Batchgröße bleibt erhalten, Conv ändert die Kanalzahl, Pooling
# verkleinert nur die räumlichen Dimensionen.
planned_shapes = [
    (16, 28, 28, 1),             # Eingabe
    (16, 28, 28, 8),             # Conv, same
    (16, 14, 14, 8),             # MaxPool 2
    (16, 12, 12, 16),            # Conv 3, valid
    (16, 6, 6, 16),              # MaxPool 2
    (16, 6 * 6 * 16),            # Flatten
]
print("\nGeplante Tensorformen:")
for shape in planned_shapes:
    print(shape)

### Reflexion zu Aufgabe 5

Padding steuert, ob Randinformationen verloren gehen. Bei ungeradem Kernel und Stride 1 hält `same` die räumliche Größe, während `valid` sie verkleinert. Pooling reduziert Auflösung und Rechenaufwand, verwirft aber Details. Die Zahl der Feature Maps entspricht der Zahl der Filter. Beim Flatten werden alle räumlichen Positionen und Kanäle pro Beispiel zu einem Vektor zusammengeführt, die Batchachse bleibt erhalten.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.